# SALM Fine-tuning 14 · Grading the accent, kindly: pronunciation assessment (speechocean762)

> **Task** read-aloud sentence → four 0-10 scores (accuracy, fluency, prosody, total) · **Data** speechocean762
> **Metric** Pearson r of the *total* score ↑ · **Result** r 0.217 → **0.678** (+213%) · **Time** ~1 h

Part of the [SALM fine-tuning series](README.md); see [00_Overview_and_Setup](00_Overview_and_Setup.ipynb) first.

## The problem: a teacher's ear

speechocean762 has 5,000 English sentences read by Mandarin-L1 speakers, children and adults. Five experts scored each
from 0 to 10 for **accuracy** (right sounds), **fluency** (no stumbling), **prosody** (natural rhythm and intonation)
and **total**. Pronunciation tutors, language apps and reading assessment all need this.

We ask the model to write its scores as plain text, `accuracy 8, fluency 9, prosodic 8, total 8`, and correlate its
numbers with the experts' (Pearson r). The headline is the *total* score, fixed before looking. The base model's
fluency ratings already track the experts a little, but its total score barely does. The goal is scores that
track the experts on speakers the model has never heard.

In [ ]:
import json, os, shlex, subprocess, sys, time
from pathlib import Path


def find_nemo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "examples" / "speechlm2" / "finetuning").is_dir():
            return candidate
    raise RuntimeError(f"Could not locate the NeMo source tree from {start}. Open this notebook from inside the repo checkout.")


NEMO_ROOT = find_nemo_root(Path.cwd())
FT = NEMO_ROOT / "examples" / "speechlm2" / "finetuning"              # the tools used below
WORK = Path(os.environ.get("SALM_FT_WORK", NEMO_ROOT / "tutorials" / "speechlm2" / "finetuning" / "workdir")).absolute()
os.environ["SALM_FT_WORK"] = str(WORK)                                # data, manifests, experiments, exports
os.environ.pop("MPLBACKEND", None)  # the kernel's inline plotting backend would break matplotlib in child processes
if not os.environ.get("SALM_CHECKPOINT"):
    raise RuntimeError(
        "SALM_CHECKPOINT is not set. Point it at your SALM checkpoint: a directory in HuggingFace format "
        "exported by SALMAutomodel. Run `export SALM_CHECKPOINT=/path/to/checkpoint` before "
        "starting Jupyter, or set os.environ['SALM_CHECKPOINT'] in a cell above this one."
    )
CKPT = Path(os.environ["SALM_CHECKPOINT"]).expanduser().absolute()  # base SALM checkpoint (HuggingFace format)
if not CKPT.is_dir():
    raise RuntimeError(f"SALM_CHECKPOINT={CKPT} is not a directory. It must be a SALM checkpoint in HuggingFace format.")
PYTHON = os.environ.get("TRAIN_PYTHON", "python")                     # NeMo training environment
VLLM_PYTHON = os.environ.get("VLLM_PYTHON", "python")                 # vLLM 0.28 environment (ideally a separate one)
M, EXP, EXPORT, EVAL = (WORK / d for d in ("manifests", "experiments", "exports", "eval"))
q = shlex.quote                                                        # shell-quote prompts (keeps non-ASCII intact)
for d in (M, EXP, EXPORT, EVAL, WORK / "logs", WORK / "configs"):
    d.mkdir(parents=True, exist_ok=True)


RETRIES = int(os.environ.get("SALM_FT_RETRIES", 3))                   # raise when other jobs share the GPU


def sh(cmd, log=None, retries=1):
    """Run a shell command from the NeMo root. With `log`, output goes to a file and only its tail is shown."""
    for attempt in range(1, retries + 1):
        print(f"$ {cmd}" + (f"   > {log}" if log else ""), flush=True)
        if log:
            with open(WORK / "logs" / log, "w") as f:
                rc = subprocess.run(cmd, shell=True, cwd=NEMO_ROOT, stdout=f, stderr=subprocess.STDOUT).returncode
            tail = (WORK / "logs" / log).read_text(errors="ignore").splitlines()
            print("\n".join(l for l in tail[-400:] if l.startswith(("[eval]", "[export]", "[scores]", "[merge]", "wrote")))[-3000:])
        else:
            rc = subprocess.run(cmd, shell=True, cwd=NEMO_ROOT).returncode
        if rc == 0:
            return
        if attempt < retries:  # e.g. a vLLM engine starting while another job allocates GPU memory
            print(f"(attempt {attempt} failed, retrying in 90 s)")
            time.sleep(90)
    raise RuntimeError(f"command failed (rc={rc}): {cmd}" + (f"; see {WORK / 'logs' / log}" if log else ""))


def evaluate(model, manifest, name, task_args, extra=""):
    """Decode `manifest` with vLLM (greedy) and return the metric summary.

    Re-running a cell reuses a previous result only if its fingerprint (model, manifest, prompt, decoder and scoring
    settings, code revision) matches; a result produced from different inputs is an error, never silently reused.
    """
    out = EVAL / f"{name}.jsonl"
    sh(f"{VLLM_PYTHON} {FT}/vllm_task_eval.py --model {model} --manifest {manifest} --out {out} {task_args} {extra} "
       f"--reuse", log=f"{name}.log", retries=RETRIES)
    return json.load(open(f"{out}.summary.json"))


def tidy(keep, name):
    """Free disk: delete this tutorial's other merged exports (~63 GB each) and leftover training checkpoints."""
    import shutil

    for p in EXPORT.glob(f"{name}_*"):
        if p != keep:
            shutil.rmtree(p)
    shutil.rmtree(EXP / name / "checkpoints", ignore_errors=True)
    print(f"kept {keep}; removed the other {name} exports and checkpoints")


def samples(name, n=3, keys=("text", "pred_text")):
    """Print a few decoded examples."""
    rows = [json.loads(l) for l in open(EVAL / f"{name}.jsonl")]
    for r in rows[:n]:
        print("  |  ".join(f"{k}: {str(r.get(k))[:110]}" for k in keys))


def result(name, value, expected, tol):
    """Compare a number with the expected result, and record it in results.json."""
    ok = abs(value - expected) <= tol
    print(f"{'✅' if ok else '❌'} {name}: {value:.4f}  (expected: {expected} ± {tol})")
    res = WORK / "results.json"
    d = json.loads(res.read_text()) if res.exists() else {}
    d[name] = {"value": value, "expected": expected, "tolerance": tol, "ok": ok}
    res.write_text(json.dumps(d, indent=1))
    return ok


print("NeMo root:", NEMO_ROOT, "\nwork dir: ", WORK, "\ncheckpoint:", CKPT)

## 1. Data

The training set is split by speaker into train and dev; the official test set has 2,500 sentences from other speakers.
The prompt includes the sentence being read, so the model can judge *how* it was said.

In [ ]:
d = M / "speechocean"
sh(f"{PYTHON} {FT}/prepare_benchmarks.py speechocean", log="prep_so.log")
sh(f"{PYTHON} {FT}/prepare_benchmarks.py librispeech", log="prep_ls.log")
r = json.loads(open(d / "train.json").readline())
print(r["context"], "\n→", r["text"])

## 2. Baseline

In [ ]:
TASK = "--task pcc --max-tokens 48"
P2 = "{context} Use the full 0-10 range; 10 means native-like pronunciation."
base = {}
for p, extra in (("p1", ""), ("p2", f"--prompt-template {q(P2)}")):
    for s in ("dev", "test"):
        base[p, s] = evaluate(CKPT, d / f"{s}.json", f"pron_base_{p}_{s}", TASK, extra)
    print(f"{p}  dev r {base[p, 'dev']['score']:.3f}   test r {base[p, 'test']['score']:.3f}   "
          f"(test, all aspects: {({k: round(v, 3) for k, v in base[p, 'test']['pcc'].items()})})")
BEST_P = max(("p1", "p2"), key=lambda p: base[p, "dev"]["score"])  # prompt chosen on dev
BASELINE = base[BEST_P, "test"]["score"]  # its test score: the baseline fine-tuning is compared with
ORACLE = max(base[p, "test"]["score"] for p in ("p1", "p2"))  # best prompt chosen on test: an upper bound
print(f"baseline prompt (chosen on dev): {BEST_P}; test-oracle (best prompt on test): {ORACLE:.2f}")
print(f"\nbaseline r = {BASELINE:.3f}")
pairs = " ".join(f"{d}/{s}.json {d}/{s}_ft.json" for s in ("train", "dev", "test"))
sh(f"{PYTHON} {FT}/set_context.py --template {q(P2)} {pairs}")

## 3. Config: regression, written as text

Nothing special is needed. The target is the four scores in a fixed format, trained with the ordinary next-token loss,
and the metric parses the numbers back leniently (`Prosody: 6` works too).

In [ ]:
CFG = WORK / "configs" / "pron.yaml"
sh(f"{PYTHON} {FT}/make_ft_config.py --checkpoint {CKPT} --prompt manifest --out {CFG} --exp-dir {EXP / 'pron'} "
   f"--train-manifest {d}/train_ft.json --val-manifest {d}/dev_ft.json --val-name pron_dev "
   f"--train-encoder --train-proj --lr 1e-4 --warmup-steps 30 --max-steps 300 "
   f"--batch-tokens 12000 --limit-train-batches 22 --val-every-n-epochs 2 --limit-val-batches 38 --save-top-k -1")

## 4. Train (~20 min)

In [ ]:
sh(f"{PYTHON} {FT}/run_training.py --config {CFG} --preflight --smoke --keep-steps 220 264")

## 5. Select on dev

Expected dev r (total): **step 220 → 0.773**, step 264 → 0.759.

In [ ]:
dev = {}
for step in (220, 264):
    out = EXPORT / f"pron_s{step}"
    sh(f"{PYTHON} {FT}/export_checkpoint.py --exp-dir {EXP / 'pron'} --step {step} --base-checkpoint {CKPT} "
       f"--out {out} --delete-ckpt", log=f"export_pron_{step}.log")
    dev[step] = evaluate(out, d / "dev_ft.json", f"pron_s{step}_dev", TASK)["score"]
    print(f"step {step}: dev r (total) {dev[step]:.3f}")
BEST = max(dev, key=dev.get)

## 6. Test, once

In [ ]:
MODEL = EXPORT / f"pron_s{BEST}"
ft = evaluate(MODEL, d / "test_ft.json", "pron_TEST", TASK)
guard = evaluate(MODEL, M / "librispeech" / "test_clean.json", "pron_librispeech", "--task asr --normalizer english")
print(f"test r (total): base {BASELINE:.3f} → fine-tuned {ft['score']:.3f} ({(ft['score'] - BASELINE) / BASELINE:+.1%})")
print("all aspects:", {k: round(v, 3) for k, v in ft["pcc"].items()}, f"| LibriSpeech WER {100 * guard['score']:.2f}%")
samples("pron_TEST", n=4, keys=("sentence", "text", "pred_text"))
result("14 baseline test r", BASELINE, 0.217, 0.01)
result("14 fine-tuned test r", ft["score"], 0.678, 0.03)

Expected results:

| r on test | accuracy | fluency | prosody | **total** |
|---|---:|---:|---:|---:|
| base, prompt chosen on dev (p2) | 0.202 | 0.412 | 0.384 | 0.217 |
| **fine-tuned** | 0.608 | 0.747 | 0.726 | **0.678** |

**+213% relative** on the total score, and every aspect improves. The prompt chosen on dev (p2) is also the
test-oracle. Systems built specifically for this dataset report total-score correlations of roughly 0.7-0.8, so a
general speech LLM gets within reach after 20 minutes of training. LibriSpeech WER rises from 1.54% to ~2.02%: judging
accents slightly dulls transcribing them, one of the larger side effects in the series.

## Tidy up

Keep the selected model; delete the other exports (~63 GB each) and any leftover training checkpoints.

In [ ]:
tidy(keep=MODEL, name="pron")

## Takeaways

1. **Numbers can be text.** Write the scores in a fixed format and parse leniently.
2. **Fix your headline metric before you look.** Total is one of the two weakest aspects for the base model; switching
   to another aspect after seeing the results would change the size of the gain.
3. **Watch the guard.** Tasks that repurpose the encoder cost a little ASR.